In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import gc
import pandas as pd
import pyarrow.parquet as pq

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV

In [3]:
pf = pq.ParquetFile(r'../prep_data/data_train_std.pq')
for i in range(pf.num_row_groups):
    num_rows_in_rg = pf.metadata.row_group(i).num_rows
    print(f'Row group {i} содержит {num_rows_in_rg} строк')

Row group 0 содержит 1048576 строк
Row group 1 содержит 1048576 строк
Row group 2 содержит 302848 строк


In [4]:
table = pf.read_row_group(0)
batch = table.to_pandas()
X_train = batch.drop(['id', 'flag'], axis=1)
y_train = batch['flag']

In [5]:
X_train.shape, y_train.shape

((1048576, 386), (1048576,))

In [6]:
tf = pq.ParquetFile(r'../prep_data/data_test_std.pq')
for i in range(tf.num_row_groups):
    num_rows_in_rg = tf.metadata.row_group(i).num_rows
    print(f'Row group {i} содержит {num_rows_in_rg} строк')

Row group 0 содержит 600000 строк


In [7]:
table = tf.read_row_group(0)
batch = table.to_pandas()
X_test = batch.drop(['id', 'flag'], axis=1)
y_test = batch['flag']

In [8]:
X_test.shape, y_test.shape

((600000, 386), (600000,))

In [9]:
y_test

0         0
1         0
2         0
3         0
4         0
         ..
599995    0
599996    0
599997    0
599998    0
599999    0
Name: flag, Length: 600000, dtype: int64

### Обучение базовой модели (с параметрами по умолчанию)

In [10]:
rfc = RandomForestClassifier()

In [11]:
%%time
rfc.fit(X_train[:300000], y_train[:300000])

CPU times: user 2min 56s, sys: 714 ms, total: 2min 57s
Wall time: 2min 57s


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [55]:
new_order = X_train.columns

In [56]:
X_test = X_test[new_order]

In [14]:
predicted_test_rfc = rfc.predict_proba(X_test)

In [15]:
predicted_test_rfc

array([[0.87, 0.13],
       [1.  , 0.  ],
       [0.94, 0.06],
       ...,
       [1.  , 0.  ],
       [0.99, 0.01],
       [0.99, 0.01]])

In [16]:
predicted_test_rfc.shape

(600000, 2)

In [17]:
predicted_test_rfc[:, 1]

array([0.13, 0.  , 0.06, ..., 0.  , 0.01, 0.01])

In [18]:
auc = roc_auc_score(y_test, predicted_test_rfc[:, 1])
print('На базовой модели RandomForestClassifier метрика AUC равна ', auc)

На базовой модели RandomForestClassifier метрика AUC равна  0.6884409105967877


### Выполним подбор параметров модели с помощью GridSearch

In [61]:
rfc_tumed = RandomForestClassifier()

In [70]:
params = {
    'n_estimators': [800, 900, 1000],
    'max_depth' : [70, 80, 90],
    'min_samples_leaf': [20, 30, 40],
    'max_features' : ['sqrt'], 
    'class_weight': ['balanced']
}

gs = GridSearchCV(rfc_tumed, params, cv=3, scoring='roc_auc', n_jobs=2, verbose=1)

Поскольку обучение занимает продолжительное время, выполняю подбор параметров на фрагменте тренировочной выборки

In [71]:
%%time
gs.fit(X_train[:100000], y_train[:100000])

Fitting 3 folds for each of 27 candidates, totalling 81 fits
CPU times: user 3min 49s, sys: 10.1 s, total: 4min
Wall time: 1h 35min 8s


,estimator,RandomForestClassifier()
,param_grid,"{'class_weight': ['balanced'], 'max_depth': [70, 80, ...], 'max_features': ['sqrt'], 'min_samples_leaf': [20, 30, ...], ...}"
,scoring,'roc_auc'
,n_jobs,2
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,1000


In [72]:
gs.best_estimator_

,n_estimators,1000
,criterion,'gini'
,max_depth,80
,min_samples_split,2
,min_samples_leaf,40
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [73]:
gs.best_score_

0.712053752865088

In [69]:
model = gs.best_estimator_

C n_estimators>700 уже не происходит сколько нибудь значимого роста качества модели.
После подбора параметров обучаю модель на все фрагменте тренировочных данных.

In [52]:
%%time
model.fit(X_train, y_train)

CPU times: user 59min 48s, sys: 5.46 s, total: 59min 53s
Wall time: 59min 54s


,n_estimators,700
,criterion,'gini'
,max_depth,80
,min_samples_split,2
,min_samples_leaf,30
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [57]:
predicted_tuned_rfc = model.predict_proba(X_test)

In [74]:
auc_tuned = roc_auc_score(y_test, predicted_tuned_rfc[:, 1])
print('После подбора параметров RandomForestClassifier метрика AUC равна ', auc_tuned)

После подбора параметров RandomForestClassifier метрика AUC равна  0.736798004032698
